# Transformación de Datos y Pipeline

En esta sección se realiza la transformación del dataset limpio, incluyendo:
- Escalado de variables numéricas
- Codificación de variables categóricas
- Construcción de un pipeline para automatizar el proceso

#Imports necesarios

In [7]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#Petición del dataset LIMPIO al repositorio en github y su visualización

In [6]:
url = "https://raw.githubusercontent.com/Nicossm/parcial1/main/data/processed/clean.csv"
df = pd.read_csv(url)

df.head(5)

,experience,country,education,languages,frameworks,company_size,salary_usd,num_languages,num_frameworks,salary_usd_before_cap
0,24,India,Bachelors,"Go, Ruby","Django, Flask",5000+,122189.0,2,2,122189.0
1,38,France,Masters,"Python, Ruby",Unknown,201-1000,147754.0,2,1,147754.0
2,36,USA,Some College,"C++, PHP","Ruby on Rails, Spring",5000+,220819.0,2,2,220819.0
3,3,India,Masters,"Java, JavaScript","Ruby on Rails, Spring",1-10,31943.0,2,2,31943.0
4,18,USA,Masters,"Go, Java","Django, React",5000+,166442.0,2,2,166442.0


## 2. Separación de variables objetivo y predictoras

Se define la variable objetivo (`salary_usd`) y las variables predictoras.
También se eliminan variables no útiles o redundantes.

In [8]:
y = df['salary_usd']


X = df.drop(columns=[
    'salary_usd',
    'salary_usd_before_cap',
    'languages',
    'frameworks'
])

## 3. Clasificación de variables

Se separan variables numéricas y categóricas para aplicar transformaciones adecuadas.

In [10]:
num_cols = ['experience', 'num_languages', 'num_frameworks']
cat_cols = ['country', 'education', 'company_size']

## 4. Construcción del pipeline

Se aplica:
- Estandarización a variables numéricas
- OneHotEncoding a variables categóricas

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

## 5. Aplicación de transformaciones

Se ajusta el pipeline y se transforma el dataset a formato numérico.

In [13]:
X_transformed = pipeline.fit_transform(X)

## 6. Reconstrucción del dataset procesado

Se reconstruye el dataset con nombres de variables para mejorar la interpretabilidad.

In [29]:
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

X_transformed_df = pd.DataFrame(
    X_transformed.toarray(),
    columns=feature_names
)

## 7. Limpieza de nombres de columnas

Se eliminan prefijos para mejorar la lectura del dataset final.

In [31]:
X_transformed_df.columns = [
    col.replace("num__", "").replace("cat__", "")
    for col in X_transformed_df.columns
]

## 8. Exportación del dataset procesado

Se guarda el dataset final para uso en etapas posteriores del proyecto.

In [34]:
import os

os.makedirs("data/processed", exist_ok=True)

X_transformed_df.to_csv("data/processed/transformed.csv", index=False)